In [1]:
import redis.asyncio as redis  
import asyncio
import nest_asyncio
nest_asyncio.apply()

from datetime import datetime

from alpaca.data.live.stock import StockDataStream
import os 

stock_stream = StockDataStream(os.environ['API_KEY'], os.environ['SECRET_KEY'])
tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

async def push_ohlc_data(bar):
    bar = {k: v for k, v in bar}
    bar['timestamp'] = bar['timestamp'].isoformat()
    # Add the new OHLC tick to the Redis Stream
    await redis_client.xadd(f"alpaca_{bar['symbol']}", bar)

    # Trim the stream to keep only the last 30 ticks
    await redis_client.xtrim(f"ohlc_stream:{bar['symbol']}", maxlen=100)
    print(f"Pushed OHLC Tick: {bar}")


In [ ]:
stock_stream.subscribe_bars(push_ohlc_data, *tickers)
stock_stream.run()

Pushed OHLC Tick: {'symbol': 'NVDA', 'timestamp': '2025-04-15T14:04:00+00:00', 'open': 111.92, 'high': 111.96, 'low': 111.79, 'close': 111.84, 'volume': 5044.0, 'trade_count': 54.0, 'vwap': 111.850168}
Pushed OHLC Tick: {'symbol': 'INTC', 'timestamp': '2025-04-15T14:04:00+00:00', 'open': 20.325, 'high': 20.325, 'low': 20.3, 'close': 20.305, 'volume': 3627.0, 'trade_count': 36.0, 'vwap': 20.304797}
Pushed OHLC Tick: {'symbol': 'GOOG', 'timestamp': '2025-04-15T14:04:00+00:00', 'open': 160.63, 'high': 160.69, 'low': 160.6, 'close': 160.63, 'volume': 860.0, 'trade_count': 11.0, 'vwap': 160.63432}
Pushed OHLC Tick: {'symbol': 'AAPL', 'timestamp': '2025-04-15T14:04:00+00:00', 'open': 202.555, 'high': 202.63, 'low': 202.09, 'close': 202.12, 'volume': 5212.0, 'trade_count': 72.0, 'vwap': 202.367558}
Pushed OHLC Tick: {'symbol': 'AMD', 'timestamp': '2025-04-15T14:04:00+00:00', 'open': 96.11, 'high': 96.11, 'low': 95.97, 'close': 95.97, 'volume': 1024.0, 'trade_count': 18.0, 'vwap': 96.025075}
P